# Intelli-Speech — TEMPORAL_V1 (Phase 3: DSP timing module validation)

This checks that the timing module turns real typical Vaani clips into speech with the timing pattern measured in Experiment 1. That pattern is slower articulation and more, longer and more variable pauses.

The check runs once for each of the 7 leave-one-speaker-out folds:

- **Profiles** are fitted from the other 6 atypical speakers, never from the held-out one.
- **Calibration:** does each synthetic profile match the real speaker it came from?
- **Coverage:** does the held-out real speaker fall inside the range the synthetic speakers span?
- **Pitch:** is it preserved? The output/input F0 ratio should be about 1.

The inputs are read-only: the Exp 1 `features.csv` and `synth_source_v1_full/`. Speakers appear only as S1–S7. Results are written to `analysis_v2/temporal_validation_<date>/`. **Do not commit this notebook with outputs.**

In [5]:
# CELL 1 | LOGIN + SYNC + DEPENDENCIES
import os, subprocess, sys, json
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
assert Path("scripts/validate_temporal_transform.py").is_file(), "Pull the latest code first."
!pip install -q soundfile praat-parselmouth "librosa==0.10.2.post1" "numpy<2.5"
DATA_ROOT = Path("/content/drive/MyDrive/Intelli-Speech-data")
FEATURES = sorted((DATA_ROOT / "analysis_v2").glob("exp1_features_*/features.csv"))[-1]
SOURCE = DATA_ROOT / "synth_source_v1_full"
assert (SOURCE / "quality.csv").is_file(), "synth_source_v1_full/quality.csv missing"
print("features:", FEATURES.parent.name, "| source:", SOURCE.name)

features: exp1_features_2026-11-22 | source: synth_source_v1_full


In [6]:
# CELL 2 | RUN VALIDATION (7 folds x 120 clips; ~20-40 min CPU) — module v0.2
!python scripts/validate_temporal_transform.py --features "{FEATURES}" --source-dir "{SOURCE}" --data-root "{DATA_ROOT}"

[fold S1] 1/7 done (6 profiles)
[fold S2] 2/7 done (6 profiles)
[fold S3] 3/7 done (6 profiles)
[fold S4] 4/7 done (6 profiles)
[fold S5] 5/7 done (6 profiles)
[fold S6] 6/7 done (6 profiles)
[fold S7] 7/7 done (6 profiles)
{
 "created_utc": "2026-11-26T13:05:00.685295+00:00",
 "transform_version": "temporal-0.2",
 "features": "/content/drive/MyDrive/Intelli-Speech-data/analysis_v2/exp1_features_2026-11-22/features.csv",
 "source_dir": "/content/drive/MyDrive/Intelli-Speech-data/synth_source_v1_full",
 "clips_per_fold": 120,
 "folds": 7,
 "transforms": 840,
 "errors": 0,
 "f0_ratio_quantiles": [
  0.9866,
  0.9991,
  1.005
 ],
 "insertion_shortfall_total": 25,
 "click_score_in_quantiles": [
  26.21,
  66.03,
  149.43
 ],
 "click_score_out_quantiles": [
  20.5,
  45.81,
  136.7
 ],
 "click_increase_gt_1p5x_fraction": 0.007,
 "calibration_median_abs_rel_error_by_feature": {
  "articulation_rate_nuclei_per_s": 0.11,
  "pause_iqr_s_150ms": 0.498,
  "pause_iqr_s_250ms": 0.469,
  "pause_medi

In [7]:
# CELL 3 | RESULTS (pseudonymous speakers)
import pandas as pd
OUT = max((DATA_ROOT / "analysis_v2").glob("temporal_validation_*"), key=lambda p: p.stat().st_mtime)  # newest run
print("results:", OUT.name)
summary = json.loads((OUT / "summary.json").read_text())
for k in ("transform_version", "transforms", "errors", "f0_ratio_quantiles", "insertion_shortfall_total",
          "click_score_in_quantiles", "click_score_out_quantiles", "click_increase_gt_1p5x_fraction",
          "calibration_outside_control_band_rate", "coverage_rate_overall"):
    print(f"{k:40s} {summary[k]}")
print(); print("calibration |relative error| (median) by feature:")
print(pd.Series(summary["calibration_median_abs_rel_error_by_feature"]).sort_values().to_string())
print(); print("coverage of held-out speaker by feature:")
print(pd.Series(summary["coverage_rate_by_feature"]).sort_values().to_string())
print(); print("coverage by fold:"); print(pd.Series(summary["coverage_rate_by_fold"]).to_string())
cal = pd.read_csv(OUT / "calibration.csv")
print(); print("synthetic vs real (median over folds) for key features:")
key = ["articulation_rate_nuclei_per_s", "pause_rate_per_min_250ms", "pause_p90_s_250ms", "speech_activity_ratio"]
print(cal[cal.feature.isin(key)].groupby(["feature", "profile_source"])[["synthetic", "real_target"]].median().round(3).to_string())

results: temporal_validation_temporal-0.2_2026-11-26
transform_version                        temporal-0.2
transforms                               840
errors                                   0
f0_ratio_quantiles                       [0.9866, 0.9991, 1.005]
insertion_shortfall_total                25
click_score_in_quantiles                 [26.21, 66.03, 149.43]
click_score_out_quantiles                [20.5, 45.81, 136.7]
click_increase_gt_1p5x_fraction          0.007
calibration_outside_control_band_rate    0.923
coverage_rate_overall                    0.429

calibration |relative error| (median) by feature:
pause_rate_per_min_250ms          0.056
speech_activity_ratio             0.102
articulation_rate_nuclei_per_s    0.110
pause_median_s_500ms              0.112
pause_time_fraction_250ms         0.129
pause_rate_per_min_500ms          0.132
pause_time_fraction_150ms         0.162
pause_time_fraction_500ms         0.166
speech_rate_nuclei_per_s          0.188
pause_p90_s_150ms 

In [8]:
# CELL 4 | LISTEN: per example -> input, stretch only, pauses only, both
# Note which variant sounds wrong (e.g. "S3_01 stretch_only: robotic"); that tells Claude which part to fix.
from IPython.display import Audio, display
examples = sorted((OUT / "examples").glob("*.wav"))
for stem in sorted({"_".join(p.name.split("_")[:2]) for p in examples})[:8]:
    print("=" * 10, stem)
    for wav in sorted((OUT / "examples").glob(f"{stem}_*.wav")):
        print(wav.name); display(Audio(str(wav)))

========== S1_00
S1_00_0_input.wav


S1_00_1_stretch_only.wav


S1_00_2_pauses_only.wav


S1_00_3_both.wav


========== S1_01
S1_01_0_input.wav


S1_01_1_stretch_only.wav


... (22 outputs left out)

========== S2_03
S2_03_0_input.wav


S2_03_1_stretch_only.wav


S2_03_2_pauses_only.wav


S2_03_3_both.wav


Save the notebook and tell Claude. Do not commit it with outputs.